# Actual LoRA + RAG Architecture

This notebook demonstrates a realistic workflow:

- LoRA provides behavior/personality/domain specialization.
- RAG provides dynamic knowledge.
- The final prompt combines retrieved context with the LoRA-tuned model.

Example scenario:

- Base model: Qwen
- LoRA: Embedded Linux expert
- RAG: RK3588 documentation


## Architecture

```plantuml
@startuml
actor User
rectangle Retriever
rectangle Documents
rectangle LoRA_Model

User -> Retriever : Question
Retriever -> Documents : Search
Documents --> Retriever : Context
Retriever -> LoRA_Model : Prompt + Context
LoRA_Model --> User : Answer
@enduml
```


## Step 1: Load LoRA Model (GPU Training Side)

Typical Python code used before deployment.


In [ ]:
from transformers import AutoModelForCausalLM
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-3B')

model = PeftModel.from_pretrained(
    base_model,
    'embedded-linux-lora'
)


## Step 2: Tiny RAG Knowledge Base

In production this may come from PDFs, markdown, source code, or manuals.

In [ ]:
documents = [
    {
        'title': 'RK3588 NPU',
        'text': 'RK3588 provides up to 6 TOPS AI performance.'
    },
    {
        'title': 'RKLLM',
        'text': 'RKLLM executes quantized language models on Rockchip NPUs.'
    }
]

## Step 3: Retrieval

Very simple keyword retrieval.


In [ ]:
query = 'How much AI performance does RK3588 provide?'

context = ''

for doc in documents:
    if 'RK3588' in doc['text']:
        context += doc['text'] + '\n'

print(context)


## Step 4: Prompt Construction

The retrieved context is injected into the prompt.


In [ ]:
prompt = f'''
You are an embedded Linux expert.

Use only the supplied context.

Context:
{context}

Question:
{query}

Answer:
'''

print(prompt)


## Step 5: Inference

When deployed on RK3588:

1. Train LoRA on GPU.
2. Merge LoRA.
3. Convert merged model to RKLLM.
4. Use the same RAG retrieval process.
5. Send prompt to RKLLM runtime.


## RK3588 Runtime Equivalent

Pseudo-flow:

```cpp
context = retrieve(question);
prompt = build_prompt(context, question);
rkllm_run(prompt);
```

The LoRA behavior is already baked into the merged RKLLM model.


## Why This Works

RAG supplies current knowledge.

LoRA supplies expertise, formatting, style, and domain behavior.

Together they solve different problems:

- RAG answers: What should the model know?
- LoRA answers: How should the model behave?
